# BitNet Studio — Fine-tune QLoRA PT-BR + Tool-calling\n\n**GPU**: T4 (16GB VRAM) — gratuito no Google Colab\n**Modelo**: Falcon3-3B-Instruct\n**Técnica**: QLoRA 4-bit (bitsandbytes)\n**Dataset**: 61 exemplos de tool-calling em PT-BR

In [ ]:
# 1. Instalar dependências\n!pip install -q transformers peft datasets accelerate bitsandbytes safetensors

In [ ]:
# 2. Clonar repo e carregar dataset\n!git clone https://github.com/peder1981/BitNet.git /content/BitNet\nimport json\nfrom datasets import Dataset\n\nDATASET = '/content/BitNet/bitnet-studio/data/ptbr_tools_train.jsonl'\nrows = []\nwith open(DATASET, encoding='utf-8') as f:\n    for line in f:\n        if line.strip():\n            rows.append(json.loads(line))\n\ndef to_text(messages):\n    parts = []\n    for m in messages:\n        if m['role'] == 'system':\n            parts.append(f"<|system|>\n{m['content']}")\n        elif m['role'] == 'user':\n            parts.append(f"<|user|>\n{m['content']}")\n        else:\n            parts.append(f"<|assistant|>\n{m['content']}")\n    return '\n'.join(parts) + '\n<|assistant|>\n'\n\ntexts = [to_text(r['messages']) for r in rows]\nprint(f'Dataset: {len(texts)} exemplos')

In [ ]:
# 3. Configurar modelo QLoRA\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n\nMODEL = 'tiiuae/Falcon3-3B-Instruct'\nOUTPUT = '/content/f3b-ptbr-tools-qlora'\n\ntok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)\nif tok.pad_token is None:\n    tok.pad_token = tok.eos_token\n\nmodel = AutoModelForCausalLM.from_pretrained(\n    MODEL,\n    load_in_4bit=True,\n    bnb_4bit_compute_dtype=torch.bfloat16,\n    bnb_4bit_use_double_quant=True,\n    bnb_4bit_quant_type='nf4',\n    device_map='auto',\n    trust_remote_code=True,\n)\n\nmodel = prepare_model_for_kbit_training(model)\n\nlora = LoraConfig(\n    r=16, lora_alpha=32, lora_dropout=0.05,\n    bias='none', task_type='CAUSAL_LM',\n    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],\n)\nmodel = get_peft_model(model, lora)\nmodel.print_trainable_parameters()

In [ ]:
# 4. Tokenizar e treinar\nds = Dataset.from_dict({'text': texts}).map(\n    lambda b: tok(b['text'], truncation=True, max_length=256, padding=False),\n    batched=True, remove_columns=['text']\n)\n\nargs = TrainingArguments(\n    output_dir=OUTPUT + '/checkpoints',\n    max_steps=200,\n    per_device_train_batch_size=1,\n    gradient_accumulation_steps=4,\n    learning_rate=2e-4,\n    warmup_steps=20,\n    logging_steps=10,\n    save_strategy='steps', save_steps=50,\n    optim='paged_adamw_8bit',\n    fp16=False, bf16=True,\n    seed=42, report_to=[],\n)\n\ntrainer = Trainer(\n    model=model, args=args, train_dataset=ds,\n    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),\n)\n\ntrainer.train()\nmodel.save_pretrained(OUTPUT)\ntok.save_pretrained(OUTPUT)\nprint(f'Adapter salvo em: {OUTPUT}')

In [ ]:
# 5. Fazer download do adapter\nfrom google.colab import files\nimport shutil\n\nshutil.make_archive('/content/f3b-ptbr-tools-qlora', 'zip', OUTPUT)\nfiles.download('/content/f3b-ptbr-tools-qlora.zip')